In [1]:
import math

from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral
from qiskit_aer import AerSimulator
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

from qiskit import transpile
from qiskit.circuit import QuantumRegister, QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from gbasis.parsers import parse_nwchem

import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral
from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral

import scipy as sp
import pyscf

In [ ]:
dist_a = 0.7414 # internuclear distance in angstroms
dist_bohr = dist_a * 1.8897259886
mass_proton = 1874.0 #mass of proton in atomic units (electron masses)

In [8]:
# Basis dictionaries (atomic orbitals)
e_basis_dict = parse_nwchem("6-31G.nw")
n_basis_dict = parse_nwchem("DZSNB.nw")

# Electron and nucleus positions
e_pos = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, dist_bohr]])
e_atoms = ['H', 'H']

n_pos = e_pos
n_atoms = ['Q', 'Q']

# Make basis of AOs for entire molecule
e_basis = make_contractions(e_basis_dict, e_atoms, e_pos, coord_types="cartesian")
n_basis = make_contractions(n_basis_dict, n_atoms, n_pos, coord_types="cartesian")

# Symmetric orthonormalization to get orthonormal MOs
e_ovlp = overlap_integral(e_basis)
e_ortho = np.linalg.inv(sp.linalg.sqrtm(e_ovlp))

n_ovlp = overlap_integral(n_basis)
n_ortho = np.linalg.inv(sp.linalg.sqrtm(n_ovlp))

# Get kinetic energy one-body integrals
e_ke = kinetic_energy_integral(e_basis, transform=e_ortho)
n_ke = kinetic_energy_integral(n_basis, transform=n_ortho)/mass_proton

# Get coulomb interaction two-body integrals
ee_coulomb = electron_repulsion_integral(e_basis, transform=e_ortho)
nn_coulomb = electron_repulsion_integral(n_basis, transform=n_ortho)


# Combine bases to get the coulomb interaction between nuclei and electrons
ne_ortho = np.block([[e_ortho, np.zeros((len(e_basis), len(n_basis)))],
                       [np.zeros((len(n_basis), len(e_basis))), n_ortho]])

ne_basis = e_basis + n_basis

ne_coulomb = electron_repulsion_integral(ne_basis, transform=ne_ortho)

In [10]:
e_basis_size = len(e_basis) # Number of one particle states
n_basis_size = len(n_basis)

e_modes = 2*len(e_basis) # Number of modes once we consider spin
n_modes = 2*len(n_basis)

mapper = JordanWignerMapper()

# SparsePauliOp identity for qubits representing the electronic/nuclear modes
e_plop_id = SparsePauliOp("I"*e_modes)
n_plop_id = SparsePauliOp("I"*n_modes)

#
# Now construct our Hamiltonian
#

# FermionOp for Hamiltonian terms that only have electronic d.o.f.
h_fmop_e = FermionicOp({}, num_spin_orbitals=e_modes)

# Electronic KE
for i in range(len(e_basis)):
    for j in range(len(e_basis)):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        h_fmop_e += FermionicOp({f"+_{2*i} -_{2*j}": e_ke[i, j]}, num_spin_orbitals=e_modes)
        h_fmop_e += FermionicOp({f"+_{2*i+ 1} -_{2*j + 1}": e_ke[i, j]}, num_spin_orbitals=e_modes)

# Electronic coulomb-coulomb repulsion
for i in range(len(e_basis)):
    for j in range(len(e_basis)):
        for k in range(len(e_basis)):
            for l in range(len(e_basis)):
                for spin1 in range(2):
                    for spin2 in range(2):
                        h_fmop_e += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin2} -_{2*l + spin1}": 0.5 * ee_coulomb[i, j, l, k],
                        }, num_spin_orbitals=e_modes)

# FermionOp for Hamiltonian terms that only have nuclear d.o.f.
h_fmop_n = FermionicOp({}, num_spin_orbitals=n_modes)

# Nuclear KE
for i in range(len(n_basis)):
    for j in range(len(n_basis)):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        h_fmop_n += FermionicOp({f"+_{2*i} -_{2*j}": n_ke[i, j]}, num_spin_orbitals=n_modes)
        h_fmop_n += FermionicOp({f"+_{2*i+ 1} -_{2*j + 1}": n_ke[i, j]}, num_spin_orbitals=n_modes)

# Nuclear coulomb-coulomb repulsion
for i in range(len(n_basis)):
    for j in range(len(n_basis)):
        for k in range(len(n_basis)):
            for l in range(len(n_basis)):
                for spin1 in range(2):
                    for spin2 in range(2):
                        h_fmop_n += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin2} -_{2*l + spin1}": 0.5 * nn_coulomb[i, j, l, k],
                        }, num_spin_orbitals=n_modes)

# SparsePauliOp representation of h_fmop_e and h_fmop_n
h_plop_e = mapper.map(h_fmop_e) ^ n_plop_id
h_plop_n = e_plop_id ^ mapper.map(h_fmop_n)

# SparsePauliOp rep of (what will become) our full Hamiltonian!
h_plop = h_plop_e + h_plop_n

# Nuclear-electron coulomb-coulomb repulsion. I need to do this in this weird way because there doesn't seem to do a way to do two separate JWT's with JordanWignerMappper for distinguishable particles
for i in range(len(n_basis)):
    for l in range(len(n_basis)):
        for spin1 in range(2):
            h_nec_fmop_n = FermionicOp({f"+_{2*i + spin1} -_{2*l + spin1}": 1}, num_spin_orbitals=n_modes)
            for j in range(len(e_basis)):
                for k in range(len(e_basis)):
                    for spin2 in range(2):
                        h_nec_fmop_e = FermionicOp({f"+_{2*j + spin2} -_{2*k + spin2}": 1}, num_spin_orbitals=e_modes)

                        h_plop -= ne_coulomb[i, j, l, k] * mapper.map(h_nec_fmop_e) ^ mapper.map(h_nec_fmop_n)

h_mtx = h_plop.to_matrix(sparse=True)
h_mtx = np.round(h_mtx, 4)

In [ ]:
import itertools

e_num = 2
n_num = 2

valid_states = [] #binary strings in occ number basis corresponding to states with 2 electorns and 2 protons
valid_states_indices = [] #decimal version of these binary strings

for e_indices in itertools.combinations(range(e_modes), e_num):
    e_string = ['0'] * e_modes

    for i in e_indices:
        elec_string[i] = '1'

    total_string = "".join(e_string)

    valid_states.append(total_string)
    valid_states_indices.append(int(total_string, 2))

all_states_indices = list(range(2**(elec_modes)))
invalid_states_indices = [index for index in all_states_indices if index not in valid_states_indices]